# WordCount con análisis CAP: primer job Spark real

**Curso:** ST1630-2026-2 · **Semana:** S3
**Estudiante:** _(nombre completo aquí)_
**Fecha:** _(fecha de ejecución aquí)_
**Estatus:** formativo — no calificado (ver `../README.md`)

## Objetivo de la sesión

Ejecutar un job real de Apache Spark (WordCount sobre reseñas de producto)
observando el DAG, la evaluación perezosa y los shuffles en Spark UI, para
luego conectar esa evidencia empírica con el Teorema CAP (Brewer, 2000):
¿por qué Spark puede sostener una posición AP gracias al linaje del DAG,
en vez de la posición CP que adopta MapReduce al escribir el shuffle a disco?

## Parte 1: Setup y carga de datos

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# getOrCreate() funciona tanto en Databricks (donde ya existe una sesión
# activa administrada por el cluster) como en un entorno local con
# pyspark instalado (`pip install pyspark`), sin necesitar ramas de código.
spark = SparkSession.builder.appName("ST1630-S3-WordCount-Reseñas").getOrCreate()

# Elige UNA de las dos rutas según dónde estés ejecutando este notebook.
# Local (ejecutando desde notebook_base/, con el repo clonado):
ruta_datos = "../datos/reseñas_producto.txt"

# Databricks (después de subir el archivo vía Catalog -> Upload File):
# ruta_datos = "dbfs:/FileStore/reseñas_producto.txt"

reseñas_df = spark.read.text(ruta_datos)

reseñas_df.show(5, truncate=False)
print("Total de líneas leídas:", reseñas_df.count())

## Parte 2: Transformaciones (plan lazy)

In [ ]:
# Stopwords en español para este dominio (reseñas de e-commerce).
# Se define como variable propia -- no como import de una librería externa --
# para que la lista sea auditable y ajustable línea a línea por el estudiante.
STOPWORDS_ES = [
    "el", "la", "los", "las", "de", "del", "en", "y", "a",
    "que", "se", "un", "una", "con", "por", "es", "no", "al", "lo", "su",
]

palabras_df = (
    reseñas_df
    .select(F.explode(F.split(F.col("value"), r"\s+")).alias("palabra_raw"))
    .select(F.lower(F.col("palabra_raw")).alias("palabra"))
    .select(F.regexp_replace(F.col("palabra"), r"[^\wáéíóúñü]", "").alias("palabra"))
    .filter(F.col("palabra") != "")
    .filter(~F.col("palabra").isin(STOPWORDS_ES))
)

# Nada se ejecuta aún: explode, split, lower y filter son transformaciones
# lazy. Spark solo construye el plan lógico (el DAG) hasta que llegue una
# acción (show, count, collect...). Esta celda no debería imprimir ningún
# resultado de datos, solo la referencia al DataFrame resultante.
palabras_df

## Parte 3: Acción — ejecutar el DAG

In [ ]:
conteo_df = (
    palabras_df
    .groupBy("palabra")
    .count()
    .orderBy(F.col("count").desc())
)

# .show() ES la acción que dispara la ejecución de todo el plan lazy
# construido en la Parte 2. Aquí es donde Spark UI empieza a registrar
# el job, sus stages y sus shuffles.
conteo_df.show(20, truncate=False)

# El Physical Plan muestra los nodos Exchange (shuffle) y HashAggregate
# (agregación parcial antes y después del shuffle). Ubica cuántos
# Exchange aparecen antes de continuar a la Parte 4.
conteo_df.explain()

## Parte 4: Análisis del DAG en Spark UI

In [ ]:
# Cómo navegar a Spark UI en Databricks Community Edition:
#   1. Compute (menú lateral) -> clic en el nombre de tu cluster
#   2. Pestaña "Spark UI" (dentro del detalle del cluster, no en el menú lateral)
#   3. Sub-pestaña "SQL / DataFrame"
#   4. Verás una entrada por cada acción ejecutada. Abre la del job de la
#      Celda 7 (groupBy + count + orderBy) y cuenta los nodos "Exchange"
#      en el gráfico del plan.
#
# En local: la Spark UI corre en http://localhost:4040 mientras la
# SparkSession esté activa.

# Job simplificado (sin orderBy) para comparar el número de shuffles
# contra el job completo de la Celda 7.
conteo_simple_df = (
    palabras_df
    .groupBy("palabra")
    .count()
)

conteo_simple_df.show(20, truncate=False)
conteo_simple_df.explain()

# Abre este segundo job en Spark UI y compara: ¿tiene menos nodos Exchange
# que el job con orderBy? Anota ambos números -- los necesitas en la Parte 5.

## Parte 5: Análisis CAP ← ENTREGABLE PRINCIPAL

### a) Operaciones del pipeline y su tipo de shuffle

Completa con lo que observaste en tu propia ejecución (no un número "esperado").

| Operación | Tipo (shuffle / no-shuffle) | ¿Exchange en el plan físico? (sí/no) |
|---|---|---|
| `explode` (tokenización) | → [tu respuesta aquí] | → [tu respuesta aquí] |
| `split` | → [tu respuesta aquí] | → [tu respuesta aquí] |
| `lower` | → [tu respuesta aquí] | → [tu respuesta aquí] |
| `filter` (stopwords y vacíos) | → [tu respuesta aquí] | → [tu respuesta aquí] |
| `groupBy` | → [tu respuesta aquí] | → [tu respuesta aquí] |
| `count` (agregación) | → [tu respuesta aquí] | → [tu respuesta aquí] |
| `orderBy` | → [tu respuesta aquí] | → [tu respuesta aquí] |

### b) Pregunta 1 — Conteo de shuffles

¿Cuántos shuffles tiene el job completo (con `orderBy`)? ¿Y el job simplificado (sin `orderBy`)?

→ [tu respuesta aquí]

### c) Pregunta 2 — RAM vs. disco

Para cada shuffle: ¿por qué Spark elige mantenerlo en RAM en vez de escribir a disco
como MapReduce? ¿Qué garantiza el linaje del DAG que hace eso posible?

→ [tu respuesta aquí]

### d) Pregunta 3 — Clasificación CP/AP

Clasifica el WordCount de Spark como CP o AP según el Teorema CAP. Justifica en
3–5 líneas citando el comportamiento del shuffle que observaste en Spark UI.

→ [tu respuesta aquí]

### e) Pregunta bonus

¿En qué condición el WordCount de Spark se comportaría más como CP?
(pista: piensa en cuándo Spark hace *spill* a disco).

→ [tu respuesta aquí]

## Parte 6: Extensión — variaciones del pipeline

In [ ]:
# Variación a) Bigramas: contar pares de palabras consecutivas en vez de
# palabras individuales. Pista: F.window / arrays con slicing sobre la
# lista de palabras de cada línea, o F.array_join con offset de 1 posición.
# TODO: construir bigramas_df a partir de reseñas_df (antes del explode,
# necesitas la lista completa de palabras por línea, no ya explotada).
bigramas_df = None  # TODO: reemplazar por tu pipeline de bigramas


# Variación b) Filtrar solo reseñas que contengan la palabra "entrega" y
# contar las palabras más frecuentes en ese subconjunto.
# TODO: filtrar reseñas_df por F.col("value") conteniendo "entrega"
# (case-insensitive), y reaplicar el pipeline de tokenización de la Parte 2.
entrega_top_df = None  # TODO: reemplazar por tu pipeline filtrado por "entrega"


# Variación c) Promedio de palabras por reseña.
# TODO: contar palabras por línea (antes del explode) y promediar con
# F.avg sobre el DataFrame resultante.
promedio_palabras_df = None  # TODO: reemplazar por tu cálculo del promedio

## Parte 7: Bitácora de delegación

Completa la tabla según lo que realmente delegaste a un agente de IA
durante este taller (ver `../../../docs/politica-ia.md`).

| Tarea | ¿Delegado a agente? | Herramienta | Justificación |
|---|---|---|---|
| Instalar dependencias / resolver errores de setup | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |
| Adaptar las stopwords a tu criterio | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |
| Escribir el análisis CAP (preguntas 1–3 + bonus) | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |
| Interpretar el DAG en Spark UI | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |
| Resolver las variaciones de la Parte 6 | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |

> Recuerda: la lectura de Spark UI y el análisis CAP deben reflejar tu
> propio razonamiento (ver `../README.md`, sección "Bitácora de delegación").

## Entrega

1. Guarda este notebook con los outputs visibles (no lo "limpies" antes de subir).
2. Copia tu carpeta de entrega siguiendo la estructura descrita en `../README.md`
   (sección "Entregable — estructura del PR"), usando `../equipo-ejemplo/` como
   referencia de calidad.
3. Abre el Pull Request hacia `main` con título
   `taller(s3): <tu nombre> — wordcount + análisis CAP`, antes del cierre de la sesión.